# Process and trace Slakbreen data

In [1]:
%matplotlib qt

In [2]:
import numpy as np
import matplotlib.pyplot as plt
import pickle
import os
from scipy.interpolate import interp1d
import pandas as pd
from radar import ReadFile, SkyCalibration, Combine_channels, Correct_speed


# Pick surface and bed

In [3]:
#make 2d array with 0 in lower 1/3, 1 in middle 1/3 and 2 in upper 1/3
max_range = 3.5 # meters
# filename_sky_calibration = "09_03_2026_1"
# direc = os.listdir("C:\\Users\\Niklas\\Documents\\Semester Svalbard\\Snow_and_Ice_Processes\\Fieldwork\\radar_TELLBREEN_2026\\storage")
# filename = "10_03_2026_2"#direc[2]

filename = "04_03_2026_1"
filename_sky_calibration = "09_03_2026_1"
Sky_calibration = ReadFile(filename_sky_calibration)
data1 = SkyCalibration(Sky_calibration, ReadFile(filename))
data1 = Combine_channels(data1)
data1 = Correct_speed(data1, 1.2, 2.44917*(10**8)) #2.44917
data = data1['combined_power_log'].T
y = data1['x']
data = data[y<=max_range][:]
data = data.T
y = y[y<=max_range]
#y1 = y1[y1<=max_range]
x = data1['slowtime']

max_value = np.max(data, axis=1)
data = np.transpose(data)/max_value

# u,v = np.shape(data)
# data = np.random.rand(u,v)
# plt.pcolormesh(x, y, data)
# plt.gca().invert_yaxis()
# data = np.zeros((100, 100))
# data[0:33, :] = 0
# data[33:66, :] = 1
# data[66:100, :] = 2
# plt.imshow(data, aspect='auto')

('Bandwidth', 2500)
('Bandwidth', 2500)


In [4]:
#x = np.arange(1,data.shape[0])  # horizontal axis (distance)
#y = np.arange(1,data.shape[1])  # vertical axis (elevation)
block_size = 100
n_blocks = len(x) // block_size + (len(x) % block_size != 0)
surface_pts = []
for num in range(n_blocks):
    x_block = x[num * block_size : (num + 1) * block_size]
    data_block = data[:, num * block_size : (num + 1) * block_size]

    fig, ax = plt.subplots(figsize=(14, 7))

    ax.pcolormesh(
        x_block,
        y,
        data_block,
        vmin=0,
        vmax=1,
        shading="auto"  
    )

    ax.invert_yaxis()
    ax.set_title("Block " + str(num+1) + " / " + str(n_blocks) +" : Click surface points, then press Enter")

    plt.show()

    surface_pts_block = plt.ginput(n=-1, timeout=0)
    plt.close(fig)

    surface_pts_block = np.asarray(surface_pts_block, dtype=float)
    if surface_pts_block.size > 0:
        surface_pts.append(surface_pts_block)  

surface_pts = np.concatenate(surface_pts, axis=0)


# fig, ax = plt.subplots(figsize=(14, 7))
# ax.pcolormesh(
#     x,
#     y,
#     data,
#     vmin = 0,
#     vmax = 1
# )
# plt.gca().invert_yaxis()
# # ax.imshow(
# #     data,
# #     aspect="auto",
# #     #cmap="gray",
# #     clim=[0,1],
# #     extent=[x.min(), x.max(), y.max(), y.min()] 
# # )
# ax.set_title("Click guide points along the SURFACE, then press Enter")
# plt.show()

# surface_pts = plt.ginput(n=-1, timeout=0)
# plt.close(fig)

# surface_pts = np.asarray(surface_pts, dtype=float)
# #print(surface_pts)

In [5]:
# bed picks
block_size = 100
n_blocks = len(x) // block_size + (len(x) % block_size != 0)
bed_pts = []
for num in range(n_blocks):
    x_block = x[num * block_size : (num + 1) * block_size]
    data_block = data[:, num * block_size : (num + 1) * block_size]

    fig, ax = plt.subplots(figsize=(14, 7))

    ax.pcolormesh(
        x_block,
        y,
        data_block,
        vmin=0,
        vmax=1,
        shading="auto"   
    )

    ax.invert_yaxis()
    ax.set_title("Block " + str(num+1) + " / " + str(n_blocks) +" : Click bed points, then press Enter")

    plt.show()

    bed_pts_block = plt.ginput(n=-1, timeout=0)
    plt.close(fig)

    bed_pts_block = np.asarray(bed_pts_block, dtype=float)
    if bed_pts_block.size > 0:
        bed_pts.append(bed_pts_block)  

bed_pts = np.concatenate(bed_pts, axis=0)

# fig, ax = plt.subplots(figsize=(14, 7))
# ax.pcolormesh(
#     x,
#     y,
#     data,
#     vmin = 0,
#     vmax = 1
# )
# plt.gca().invert_yaxis()
# # ax.imshow(
# #     data,
# #     aspect="auto",
# #     #cmap="gray",
# #     clim=[0,1],
# #     extent=[x.min(), x.max(), y.max(), y.min()]  
# # )
# ax.set_title("Click guide points along the BED, then press Enter")
# plt.show()

# bed_pts = plt.ginput(n=-1, timeout=0)
# plt.close(fig)

# bed_pts = np.asarray(bed_pts, dtype=float)
# #print(bed_pts)

In [6]:
# sort and interpolate
surface_pts = surface_pts[np.argsort(surface_pts[:, 0])]
bed_pts = bed_pts[np.argsort(bed_pts[:, 0])]
xxx = bed_pts

# Remove accidental low points (y < 1)
bed_pts = bed_pts[bed_pts[:, 1] >= 1]


surface_f = interp1d(surface_pts[:, 0], surface_pts[:, 1],
                     bounds_error=False, fill_value="extrapolate")

bed_f = interp1d(bed_pts[:, 0], bed_pts[:, 1],
                 bounds_error=False, fill_value="extrapolate")

x_dense = np.linspace(x.min(), x.max(), data.shape[1])

surface_interp = surface_f(x_dense)
bed_interp = bed_f(x_dense)

/Users/hmann/ENTER/lib/python3.13/site-packages/scipy/interpolate/_interpolate.py:501: RuntimeWarning: invalid value encountered in divide
  slope = (y_hi - y_lo) / (x_hi - x_lo)[:, None]


In [7]:
# plot picks on radargram

fig, ax = plt.subplots(figsize=(14, 7))
# ax.imshow(
#     data,
#     aspect="auto",
#     cmap="gray",
#     extent=[x.min(), x.max(), y.max(), y.min()],
# )
ax.pcolormesh(
    x,
    y,
    data,
    vmin = 0,
    vmax = 1
)

# interpolated lines (smooth)
ax.plot(x_dense, surface_interp, 'r-', linewidth=2, label='Surface (interp)')
ax.plot(x_dense, bed_interp, 'b-', linewidth=2, label='Bed (interp)')

# overlay original picks
ax.plot(surface_pts[:, 0], surface_pts[:, 1], 'ro', alpha=0.5)
ax.plot(bed_pts[:, 0], bed_pts[:, 1], 'bo', alpha=0.5)
ax.invert_yaxis()

ax.set_title("Surface and bed picks (interpolated)")
ax.legend()
plt.show()

In [8]:
# plot ice thickness
snow_thickness = -surface_interp + bed_interp

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(x_dense, snow_thickness, 'k-', linewidth=2)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Snow thickness (m)")
ax.set_title("Snow thickness")
ax.grid(True, alpha=0.3)
print(np.mean(snow_thickness))
plt.show()


nan


In [9]:
folder = "PickData"
os.makedirs(folder, exist_ok=True)

np.savez(
    os.path.join(folder, filename + "_pickdata.npz"),
    surface_interp=surface_interp,
    bed_interp=bed_interp,
    snow_thickness=snow_thickness,
    x_dense=x_dense,
    combined_power_log=data,
    slowtime=x
)



df_1d = pd.DataFrame({
    "surface_interp": surface_interp,
    "bed_interp": bed_interp,
    "snow_thickness": snow_thickness,
    "x_dense": x_dense,
    "slowtime": x
})

df_1d.to_csv(os.path.join(folder, filename + "_pick_vars.csv"), index=False)

df_2d = pd.DataFrame(data)
df_2d.to_csv(os.path.join(folder, filename + "_combined_power_log.csv"), index=False)